# June 2019 event correction -- Stage III physical backtracking rerun

**Event/data correction only -- no algorithm redesign.** The real ocean-current/wind
files (`data/ocean.nc`, `data/wind.nc`) have been replaced with real June 2019 data
covering the true observation window. This notebook reruns Stage III (T05/T06/T09,
completely unmodified) for the SAME real 548 `ow-0008.jpg` parcels, the SAME 6-member
explicit forcing subset, the SAME `K` diffusivity, the SAME `dt`/`n_steps`, and the
SAME `master_seed`s as the original (October-anchored) run -- the ONLY change is
`t_obs` (now the TRUE observation time) and the ocean-data file/coordinate names.

Outputs are saved under `result/` (a NEW directory), never overwriting
`results/actual_data/`'s existing October-anchored outputs.

In [1]:
import os, sys, json, time
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import xarray as xr

from iste.observation import adapt_upstream_particles
from iste.environment import load_field_dataset
from iste.forcing_ensemble import ForcingEnsemble, ForcingEnsembleConfig
from iste.ensemble_run import EnsembleRunRecords, run_ensemble
from iste.lookup_storage import ParticleRecordStore

RESULT_DIR = os.path.join(REPO_ROOT, "result")
os.makedirs(RESULT_DIR, exist_ok=True)
print(f"NEW June-2019 outputs will be saved under {RESULT_DIR} (October outputs under "
      f"results/actual_data/ are left untouched)")

NEW June-2019 outputs will be saved under C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result (October outputs under results/actual_data/ are left untouched)


## 1. Load the real observation (`ow-0008.jpg`) -- true June 2019 timestamp

In [2]:
spill_data = json.load(open(os.path.join(REPO_ROOT, "data", "database_payload.json"), encoding="utf-8"))
spill = next(s for s in spill_data["spills"] if s["image_filename"] == "ow-0008.jpg")
assert spill["total_parcels"] == 548 == len(spill["parcels"])

TRUE_OBS_TIMESTAMP = np.datetime64(spill["timestamp"], "ns")
assert TRUE_OBS_TIMESTAMP == np.datetime64("2019-06-14T20:35:58", "ns"), TRUE_OBS_TIMESTAMP
print(f"image_filename : {spill['image_filename']}")
print(f"TRUE timestamp : {TRUE_OBS_TIMESTAMP}  (event/data correction target, confirmed)")
print(f"total_parcels  : {spill['total_parcels']}")

image_filename : ow-0008.jpg
TRUE timestamp : 2019-06-14T20:35:58.000000000  (event/data correction target, confirmed)
total_parcels  : 548


## 2. Load + regionally crop the new June 2019 `ocean.nc`

**Why crop:** the new `ocean.nc` covers nearly the whole Mediterranean
(`lat 30.19..45.98`, `lon -6.0..36.29`, `380x1016` grid) -- far larger than the
region relevant to this one spill's 72-hour drift. This notebook crops to the EXACT
SAME regional bounding box the original (October) `ocean.nc` already used
(`lat ~30.19..36.35`, `lon ~27.13..36.08`) -- a reused, not newly invented, spatial
extent already known to comfortably contain this spill's location plus 72h of drift.
This is a data-preparation/memory-practicality step only: it does not touch
`iste/environment.py`'s loader, any interpolation code, or any physical equation --
`load_field_dataset` (T01, unmodified) is then called normally on the cropped file.

In [3]:
RAW_OCEAN_PATH = os.path.join(REPO_ROOT, "data", "ocean.nc")
CROPPED_OCEAN_PATH = os.path.join(RESULT_DIR, "ocean_june2019_region.nc")

# same regional bounding box the original October ocean.nc already covered (reused, not new)
LAT_MIN, LAT_MAX = 30.15, 36.40
LON_MIN, LON_MAX = 27.05, 36.15

raw = xr.open_dataset(RAW_OCEAN_PATH)
print(f"raw ocean.nc: dims={dict(raw.sizes)}, lat {float(raw.lat.min()):.2f}..{float(raw.lat.max()):.2f}, "
      f"lon {float(raw.lon.min()):.2f}..{float(raw.lon.max()):.2f}, "
      f"time {raw.time.values[0]} .. {raw.time.values[-1]} ({len(raw.time)} steps)")

cropped = raw.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
print(f"cropped region: dims={dict(cropped.sizes)}, lat {float(cropped.lat.min()):.2f}..{float(cropped.lat.max()):.2f}, "
      f"lon {float(cropped.lon.min()):.2f}..{float(cropped.lon.max()):.2f}")
cropped.to_netcdf(CROPPED_OCEAN_PATH)
raw.close(); cropped.close()
print(f"saved cropped region to {CROPPED_OCEAN_PATH}")

C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\erddapy\erddapy.py)
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\erddapy\erddapy.py)
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:66: UserWarning: An error occurred while loading the GDAC data fetcher, it will not be available !
<class 

raw ocean.nc: dims={'time': 192, 'lat': 380, 'lon': 1016}, lat 30.19..45.98, lon -6.00..36.29, time 2019-06-07T00:30:00.000000000 .. 2019-06-14T23:30:00.000000000 (192 steps)
cropped region: dims={'time': 192, 'lat': 150, 'lon': 218}, lat 30.19..36.40, lon 27.08..36.12


saved cropped region to C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result\ocean_june2019_region.nc


In [4]:
ocean = load_field_dataset(CROPPED_OCEAN_PATH, ["uo", "vo"], time_dim="time", lat_dim="lat", lon_dim="lon")
print(f"loaded (cropped) ocean.nc: grid {ocean.lat.shape} x {ocean.lon.shape}, "
      f"time {ocean.time[0]} .. {ocean.time[-1]} ({len(ocean.time)} hourly steps)")

T_OBS = TRUE_OBS_TIMESTAMP
T72 = T_OBS - np.timedelta64(72, "h")
print(f"T0  (t_obs) = {T_OBS}")
print(f"T72         = {T72}")

coverage_ok = (ocean.time[0] <= T72) and (T_OBS <= ocean.time[-1])
print(f"ocean.nc covers the full [T72, T0] window: {coverage_ok}")
assert coverage_ok, "ocean.nc does not cover the required 72h backward window"
margin_before_h = float((T72 - ocean.time[0]) / np.timedelta64(1, 'h'))
margin_after_h = float((ocean.time[-1] - T_OBS) / np.timedelta64(1, 'h'))
print(f"margin before T72: {margin_before_h:.2f}h; margin after T0: {margin_after_h:.2f}h")

loaded (cropped) ocean.nc: grid (150,) x (218,), time 2019-06-07T00:30:00.000000000 .. 2019-06-14T23:30:00.000000000 (192 hourly steps)
T0  (t_obs) = 2019-06-14T20:35:58.000000000
T72         = 2019-06-11T20:35:58.000000000
ocean.nc covers the full [T72, T0] window: True
margin before T72: 116.10h; margin after T0: 2.90h


## 3. Confirm this is genuinely June 2019 data, not the old October file

In [5]:
assert str(ocean.time[0])[:7] == "2019-06", ocean.time[0]
assert str(ocean.time[-1])[:7] == "2019-06", ocean.time[-1]
assert not any(str(t)[:7] == "2019-10" for t in ocean.time), "October timestamps found in ocean.nc!"
print("confirmed: ocean.nc time axis is entirely June 2019, zero October timestamps")

confirmed: ocean.nc time axis is entirely June 2019, zero October timestamps


## 4. T07: adapt the actual 548 upstream parcels (same as original run)

In [6]:
parcels = spill["parcels"]
particle_records = [
    {
        "particle_id": str(p["id"]), "latitude": p["lat"], "longitude": p["lon"],
        "observation_pixel_id": str(p["id"]), "timestamp": T_OBS,
    }
    for p in parcels
]
adapted = adapt_upstream_particles(particle_records)
assert adapted.n_particles == 548
print(f"T07 adapted {adapted.n_particles} real particles at t_obs={T_OBS} (verified == 548)")

T07 adapted 548 real particles at t_obs=2019-06-14T20:35:58.000000000 (verified == 548)


## 5. Forcing ensemble + K (identical configuration to the original run -- not redesigned)

In [7]:
FULL_ENSEMBLE_N_MEMBERS_CONFIGURED = 50
FORCING_MEMBER_IDS_USED = (0, 1, 2, 3, 4, 5)
print(f"Configured full ensemble: {FULL_ENSEMBLE_N_MEMBERS_CONFIGURED} members. "
      f"ACTUAL members used: {FORCING_MEMBER_IDS_USED} (same explicit recorded development subset as the original run)")

ensemble_config = ForcingEnsembleConfig(n_members=FULL_ENSEMBLE_N_MEMBERS_CONFIGURED, master_seed=7)
ensemble = ForcingEnsemble(ocean, ensemble_config)
K_DIFFUSIVITY = 3.0  # m^2/s, same explicit PROTOTYPE constant as the original run
K = np.full((len(ocean.lat), len(ocean.lon)), K_DIFFUSIVITY)
print(f"K grid shape: {K.shape} (matches cropped ocean grid)")

Configured full ensemble: 50 members. ACTUAL members used: (0, 1, 2, 3, 4, 5) (same explicit recorded development subset as the original run)
K grid shape: (150, 218) (matches cropped ocean grid)


## 6. Stage III: real backward transport for all 548 real particles (June 2019 currents)

Calls `iste.ensemble_run.run_ensemble` (T09, unmodified) -- same `dt=3600s`,
`n_steps=72`, `master_seed=42` as the original run. No backward SDE is reimplemented.

In [8]:
DT_SECONDS = 3600.0
N_STEPS = 72
EXPECTED_N_RECORDS = 548 * len(FORCING_MEMBER_IDS_USED) * (N_STEPS + 1)
history_path = os.path.join(RESULT_DIR, "stage3_particle_history_june2019.npz")

print(f"Running Stage III backward transport: {adapted.n_particles} particles x "
      f"{len(FORCING_MEMBER_IDS_USED)} members x {N_STEPS} steps ({N_STEPS * DT_SECONDS / 3600 / 24:.1f} days) ...")
t0 = time.time()
records = run_ensemble(
    adapted, list(FORCING_MEMBER_IDS_USED), N_STEPS, DT_SECONDS,
    ocean, ensemble, ocean.lat, ocean.lon, K, master_seed=42,
)
t1 = time.time()
print(f"done in {t1 - t0:.1f}s ({(t1 - t0) / 60:.1f} min) -> {records.n_records} (particle, member, checkpoint) rows")
assert records.n_records == EXPECTED_N_RECORDS

Running Stage III backward transport: 548 particles x 6 members x 72 steps (3.0 days) ...


done in 763.1s (12.7 min) -> 240024 (particle, member, checkpoint) rows


## 7. Save the complete June 2019 particle history + summary

In [9]:
store = ParticleRecordStore()
store.add_records(records)
print(f"ParticleRecordStore (T11): {len(store)} records, {len(store.checkpoints())} checkpoints")

np.savez_compressed(
    history_path,
    particle_id=records.particle_id, forcing_member_id=records.forcing_member_id,
    observation_pixel_id=records.observation_pixel_id, timestamp=records.timestamp,
    lat=records.lat, lon=records.lon, log_weight=records.log_weight, valid=records.valid,
    death_time=records.death_time, death_lat=records.death_lat,
    death_lon=records.death_lon, death_reason=records.death_reason,
    survival_mark=records.survival_mark,
)
print(f"saved complete June 2019 particle history to {history_path}")

checkpoints = np.unique(records.timestamp)
summary_lines = [
    "Stage III June 2019 real particle history summary",
    f"event/data correction: t_obs corrected from 2019-10-14T23:00:00 (October anchor) to {T_OBS} (true observation)",
    f"n_particles (unique particle_id): {len(set(records.particle_id.tolist()))}",
    f"n_forcing_members: {len(FORCING_MEMBER_IDS_USED)} {FORCING_MEMBER_IDS_USED}",
    f"n_timestamps (checkpoints): {len(checkpoints)}",
    f"n_records total: {records.n_records}",
    f"T0  = {checkpoints.max()}",
    f"T72 = {checkpoints.min()}",
    "",
    "valid rows per checkpoint:",
]
for ts in checkpoints:
    mask = records.timestamp == ts
    n_valid = int(np.sum(records.valid[mask]))
    n_total = int(np.sum(mask))
    summary_lines.append(f"  {np.datetime64(ts,'ns')}: {n_valid}/{n_total} valid")
summary_text = chr(10).join(summary_lines)
print(summary_text)
with open(os.path.join(RESULT_DIR, "stage3_history_summary_june2019.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

ParticleRecordStore (T11): 240024 records, 73 checkpoints


saved complete June 2019 particle history to C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result\stage3_particle_history_june2019.npz
Stage III June 2019 real particle history summary
event/data correction: t_obs corrected from 2019-10-14T23:00:00 (October anchor) to 2019-06-14T20:35:58.000000000 (true observation)
n_particles (unique particle_id): 548
n_forcing_members: 6 (0, 1, 2, 3, 4, 5)
n_timestamps (checkpoints): 73
n_records total: 240024
T0  = 2019-06-14T20:35:58.000000000
T72 = 2019-06-11T20:35:58.000000000

valid rows per checkpoint:
  2019-06-11T20:35:58.000000000: 3288/3288 valid
  2019-06-11T21:35:58.000000000: 3288/3288 valid
  2019-06-11T22:35:58.000000000: 3288/3288 valid
  2019-06-11T23:35:58.000000000: 3288/3288 valid
  2019-06-12T00:35:58.000000000: 3288/3288 valid
  2019-06-12T01:35:58.000000000: 3288/3288 valid
  2019-06-12T02:35:58.000000000: 3288/3288 valid
  2019-06-12T03:35:58.000000000: 3288/3288 valid
  2019-06-12T04:35:58.000000000: 3288/3288 valid
  

## 8. Validation

In [10]:
checks = []
checks.append(("73 distinct checkpoints (T0..T72)", len(checkpoints) == 73))
checks.append(("T0 == true observation timestamp", checkpoints.max() == T_OBS))
checks.append(("T72 == T0 - 72h", checkpoints.min() == T72))
checks.append(("1-hour spacing throughout", bool(np.all(np.diff(np.sort(checkpoints)) == np.timedelta64(1, 'h')))))
checks.append(("no October 2019 timestamps in the new particle history", not any(str(t)[:7] == '2019-10' for t in checkpoints)))
checks.append(("n_records matches expected (548 x 6 x 73)", records.n_records == EXPECTED_N_RECORDS))
checks.append(("DT_SECONDS/N_STEPS/master_seeds unchanged from original run", (DT_SECONDS, N_STEPS) == (3600.0, 72)))
for label, ok in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks)
print()
print("Stage III June 2019 rerun: ALL VALIDATION CHECKS PASSED")

[OK] 73 distinct checkpoints (T0..T72)
[OK] T0 == true observation timestamp
[OK] T72 == T0 - 72h
[OK] 1-hour spacing throughout
[OK] no October 2019 timestamps in the new particle history
[OK] n_records matches expected (548 x 6 x 73)
[OK] DT_SECONDS/N_STEPS/master_seeds unchanged from original run

Stage III June 2019 rerun: ALL VALIDATION CHECKS PASSED
